# Урок 7. Представление вещественных чисел

8 класс · I четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/index.ipynb) · [← Урок 6](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-06.ipynb) · [Урок 8 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-08.ipynb)

---

Формат с плавающей запятой: мантисса и порядок. Нормализованная запись. Почему 0.1 + 0.2 не равно 0.3 и что с этим делать.

In [ ]:
#@title 🚀 Шаг 1. Регистрация и подготовка урока { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
ФИО = "" #@param {type:"string"}
Класс = "8А" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="08-07", name=ФИО, klass=Класс)

## Разбираемся

### Проблема дробных чисел

С целыми разобрались: разрядная сетка, дополнительный код, переполнение.
Но как записать 3,14? Или 0,000001? Или 6 000 000 000 000?

Отвести часть разрядов под целую часть, а часть под дробную — плохая идея.
Она называется **фиксированной запятой**, и её беда в негибкости: если под
дробную часть отвели 8 разрядов, то очень маленькие числа записать нельзя,
а если отвели много — не хватит на большие.

### Плавающая запятая

Решение подсказала физика. Огромные и крошечные величины давно записывают
так:

$$300\,000\,000 = 3{,}0 \cdot 10^{8} \qquad 0{,}000015 = 1{,}5 \cdot 10^{-5}$$

Число раскладывается на две части:

* **мантисса** — значащие цифры (3,0 и 1,5);
* **порядок** — степень основания (8 и −5).

Запятая не стоит на месте, а «плавает» — её положение задаётся порядком.
Отсюда и название.

### Нормализованная запись

Одно и то же число можно записать по-разному:

$$1500 = 1{,}5 \cdot 10^{3} = 15 \cdot 10^{2} = 0{,}15 \cdot 10^{4}$$

Чтобы представление было однозначным, договариваются о форме.
**Нормализованной** называют запись, в которой мантисса имеет ровно одну
значащую цифру до запятой:

$$1 \leqslant |{\text{мантисса}}| < 10$$

Значит из трёх вариантов выше правильный только первый: $1{,}5 \cdot 10^{3}$.

В компьютере то же самое, но основание не 10, а 2, и мантисса
нормализуется к диапазону от 1 до 2.

### Как это лежит в памяти

Стандарт IEEE 754, по которому работают все процессоры, раскладывает
32-разрядное вещественное число так:

```
  ┌─┬───────────┬───────────────────────────┐
  │s│  порядок  │         мантисса          │
  └─┴───────────┴───────────────────────────┘
   1     8 бит            23 бита
```

* **знак** s — один бит, 0 или 1;
* **порядок** — 8 бит, задаёт масштаб числа;
* **мантисса** — 23 бита, задаёт точность.

Отсюда сразу следуют два вывода. Диапазон огромен — примерно
от $10^{-38}$ до $10^{38}$. А вот **точность ограничена 23 разрядами**,
то есть примерно семью десятичными цифрами. Больше значащих цифр
в такое число просто не влезет.

### Почему 0,1 + 0,2 не равно 0,3

Самый известный сюрприз в программировании. Запустите на любом языке
`0.1 + 0.2` — получите `0.30000000000000004`.

Причина не в ошибке процессора, а в системе счисления. Дробь 1/10
в двоичной системе — **бесконечная периодическая**:

$$0{,}1_{10} = 0{,}0001100110011\ldots_2$$

Точно так же, как 1/3 в десятичной даёт бесконечную 0,333…
Бесконечную дробь в 23 разряда не уместить, поэтому она обрезается —
и в памяти лежит не 0,1, а очень близкое к ней число. Складывая два
приближения, получаем третье приближение, чуть-чуть не равное 0,3.

### Как с этим жить

> ⚠️ **Никогда не сравнивайте вещественные числа знаком `==`.**

Вместо этого проверяют, что разница достаточно мала:

```python
if abs(a - b) < 0.000001:
    ...числа считаем равными
```

А для денег вещественные числа не используют вовсе: суммы хранят
**в копейках целым числом**. Ни один банк не станет складывать рубли
в формате с плавающей запятой — потери на округлении за миллион операций
будут вполне ощутимыми.

## Смотрим, как это работает

### Пример 1. Знаменитая ошибка

In [ ]:
print("0.1 + 0.2 =", 0.1 + 0.2)
print("Равно ли это 0.3?", 0.1 + 0.2 == 0.3)
print()
print("А что там на самом деле:")
print(f"  0.1 = {0.1:.20f}")
print(f"  0.2 = {0.2:.20f}")
print(f"  0.3 = {0.3:.20f}")

Формат `:.20f` печатает двадцать знаков после запятой и показывает то,
что обычно скрыто. Видно, что ни 0.1, ни 0.2 не хранятся точно —
в памяти лежат ближайшие представимые числа.

### Пример 2. Правильное сравнение

In [ ]:
a = 0.1 + 0.2
b = 0.3

print("Наивно:    ", a == b)
print("Правильно: ", abs(a - b) < 1e-9)

# Тот же приём есть в стандартной библиотеке
import math
print("Через math:", math.isclose(a, b))

Запись `1e-9` означает $1 \cdot 10^{-9}$ — привычная научная нотация,
понятная Python напрямую. Такое пороговое значение называют **эпсилон**.

Функция `math.isclose` делает то же самое, но аккуратнее: она учитывает
масштаб сравниваемых чисел. Для больших чисел абсолютная разница в одну
миллиардную бессмысленна, и `isclose` это понимает.

### Пример 3. Нормализованная запись

In [ ]:
def нормализовать(число):
    """Разложить число на мантиссу и десятичный порядок."""
    if число == 0:
        return 0.0, 0
    порядок = 0
    мантисса = abs(число)
    while мантисса >= 10:
        мантисса /= 10
        порядок += 1
    while мантисса < 1:
        мантисса *= 10
        порядок -= 1
    if число < 0:
        мантисса = -мантисса
    return round(мантисса, 6), порядок


for число in [1500, 0.000015, 299792458, -0.5]:
    м, п = нормализовать(число)
    print(f"{число:>12} = {м} · 10^{п}")

Два цикла двигают запятую в нужную сторону: пока мантисса слишком велика —
делим и увеличиваем порядок, пока слишком мала — умножаем и уменьшаем.
Ровно это делает процессор, только с двойкой вместо десятки.

### Пример 4. Накопление ошибки

Одна неточность незаметна. Посмотрим, что будет, если её повторить
сто тысяч раз.

In [ ]:
сумма = 0.0
for _ in range(100000):
    сумма += 0.1

print(f"Складывали 0.1 сто тысяч раз")
print(f"Ожидали:   10000.0")
print(f"Получили:  {сумма}")
print(f"Ошибка:    {abs(сумма - 10000)}")

Ошибка невелика, но она есть — и в задачах, где важна каждая копейка
или каждый метр траектории, такое накопление недопустимо. Именно поэтому
деньги считают в копейках целыми числами.

## Пробуем сами

### Задача 1. Безопасное сравнение

Напишите функцию, которая сравнивает два вещественных числа
с точностью до `1e-9` и возвращает `True` или `False`.

In [ ]:
def почти_равны(а, б):
    return ...

In [ ]:
si.check("1", почти_равны, [
    ((0.1 + 0.2, 0.3), True),
    ((1.0, 1.0), True),
    ((1.0, 1.1), False),
    ((0.0, 0.0), True),
    ((100.0, 100.0000000001), True),
])

### Задача 2. Мантисса и порядок

Напишите функцию, которая раскладывает положительное число
на нормализованную мантиссу и десятичный порядок.
Верните список `[мантисса, порядок]`, мантиссу округлите
до 6 знаков функцией `round`.

`нормализация(1500)` → `[1.5, 3]`

In [ ]:
def нормализация(число):
    return ...

In [ ]:
si.check("2", нормализация, [
    (1500, [1.5, 3]),
    (0.015, [1.5, -2]),
    (7, [7.0, 0]),
    (0, [0.0, 0]),
])

### Задача 3. Верно ли сравнение

Что напечатает эта строка?

```python
print(0.1 + 0.2 == 0.3)
```

Впишите ответ строкой: `"да"` если `True`, `"нет"` если `False`.

In [ ]:
ответ = "???"

si.check_value("3", ответ, "dde7950114f546d0",
               hint="Вспомните, что лежит в памяти вместо 0.1 и 0.2.")

## Домашнее задание

### Домашнее задание 1. Деньги в копейках

Напишите функцию, которая складывает суммы денег **правильно** —
через целые копейки, без потери точности.

Функция получает список сумм в рублях (вещественные числа)
и возвращает итог в рублях, округлённый до копеек.

Порядок действий: каждую сумму переведите в копейки
(умножьте на 100 и округлите функцией `round` до целого),
сложите целые копейки, затем разделите на 100.

In [ ]:
def сложить_деньги(суммы):
    return ...

In [ ]:
si.check("дз1", сложить_деньги, [
    ([0.1, 0.2], 0.3),
    ([10.55, 20.45], 31.0),
    ([0.01] * 100, 1.0),
    ([], 0.0),
    ([99.99, 0.01], 100.0),
])

### Домашнее задание 2. Округление до знаков

Напишите функцию, которая округляет число до заданного количества знаков
после запятой и возвращает **строку** — так, чтобы нули на конце
не пропадали.

`в_строку(3.14159, 2)` → `"3.14"`
`в_строку(2.5, 3)` → `"2.500"`

Подсказка: формат `f"{число:.{знаков}f}"` делает ровно это.

In [ ]:
def в_строку(число, знаков):
    return ...

In [ ]:
si.check("дз2", в_строку, [
    ((3.14159, 2), "3.14"),
    ((2.5, 3), "2.500"),
    ((10, 0), "10"),
    ((0.15, 1), "0.1"),
])

### Домашнее задание 3. Сколько шагов до потери точности

Исследовательская задача. Начнём с числа 1.0 и будем прибавлять
всё меньшие и меньшие значения: 0.5, 0.25, 0.125 и так далее.
В какой-то момент прибавляемое станет настолько малым, что результат
перестанет отличаться от 1.0 — мантиссе просто не хватит разрядов.

Напишите функцию, которая возвращает количество шагов до этого момента.

Алгоритм: начните с `шаг = 1.0`, делите его пополам и проверяйте,
отличается ли `1.0 + шаг` от `1.0`. Считайте, сколько раз удалось
поделить, прежде чем разница исчезла.

In [ ]:
def шагов_до_потери():
    return ...

In [ ]:
si.check("дз3", шагов_до_потери, [
    ((), 53),
])

---

### Что означает полученное число

Пятьдесят три — это количество разрядов мантиссы в 64-разрядном
вещественном числе (52 хранимых плюс один подразумеваемый).
Вы только что измерили точность своего компьютера,
не открывая документацию — просто наблюдая, где он перестаёт различать
числа. Так работает эксперимент в информатике.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 6](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-06.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 8 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-08/urok-08.ipynb)